<a href="https://colab.research.google.com/github/ssarimm/Machine-Learning-Internship---Flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ssarimm/Machine-Learning-Internship---Flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
from pathlib import Path

# Look for CSV files in the current notebook environment
csv_files = list(Path(".").rglob("*.csv"))

print("CSV files found:", len(csv_files))

for file in csv_files[:20]:
    print(file)

CSV files found: 4
sample_data/mnist_test.csv
sample_data/mnist_train_small.csv
sample_data/california_housing_train.csv
sample_data/california_housing_test.csv


In [2]:
import pandas as pd
from pathlib import Path
import urllib.request

url = (
    "https://raw.githubusercontent.com/"
    "ssarimm/Machine-Learning-Internship---Flyrank/"
    "main/data/raw/content_refresh_anonymized.csv"
)

output_path = Path("content_refresh_anonymized.csv")

try:
    urllib.request.urlretrieve(url, output_path)
    df = pd.read_csv(output_path)

    print("Dataset loaded successfully!")
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

except Exception as e:
    print("Download failed:", e)

Dataset loaded successfully!
Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [3]:
# ML-05 — Step 1: Inspect the dataset and target

print("Dataset shape:", df.shape)

print("\nTarget distribution:")
print(df["trend_direction"].value_counts(dropna=False))
print("\nTarget distribution (%):")
print(
    (df["trend_direction"].value_counts(normalize=True, dropna=False) * 100)
    .round(2)
)

print("\nMissing values (top 15 columns):")
print(df.isna().sum().sort_values(ascending=False).head(15))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget-related columns:")
print([
    col for col in df.columns
    if any(word in col.lower() for word in ["trend", "declin", "target", "label"])
])

print("\nNumeric feature summary:")
print(df.select_dtypes(include="number").describe().round(2).T)

Dataset shape: (30000, 44)

Target distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Target distribution (%):
trend_direction
down      54.21
stable    19.87
up        14.63
new        7.45
flat       3.84
Name: proportion, dtype: float64

Missing values (top 15 columns):
provider_used        21438
word_count            7699
char_count            7699
word_count_tier       7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
cpc                   2468
competition           2468
main_intent           2374
scroll_rate            125
content_type             0
client_id                0
dtype: int64

Duplicate rows: 0

Target-related columns:
['trend_direction', 'trend_pct']

Numeric feature summary:
                          count      mean       std    min      25%       50%  \
search_volume           27532.0    158

A feature vector is the set of inputs the model uses to make a prediction. For this first version, we'll use content and keyword characteristics, not traffic metrics that may overlap with the trend calculation.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-05 — Step 2: Build the feature vector

import pandas as pd

# 1. Create a binary proxy target:
# 1 = declining content; 0 = all other trend categories
y = (df["trend_direction"] == "down").astype(int)

# 2. Choose numeric features available for consideration
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
]

# 3. Choose categorical features
categorical_features = [
    "content_type",
    "main_intent",
    "competition_level",
]

# 4. Build the raw feature table
feature_columns = numeric_features + categorical_features
X = df[feature_columns].copy()

# 5. Confirm that the target and identifiers are not features
assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns
assert "content_id" not in X.columns
assert "client_id" not in X.columns

print("Feature table shape:", X.shape)
print("Target shape:", y.shape)

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTarget counts (1 = down, 0 = not down):")
print(y.value_counts().sort_index())

print("\nMissing values in selected features:")
print(X.isna().sum())

print("\nFeature vector created successfully.")


Feature table shape: (30000, 10)
Target shape: (30000,)

Numeric features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update']

Categorical features:
['content_type', 'main_intent', 'competition_level']

Target counts (1 = down, 0 = not down):
trend_direction
0    13738
1    16262
Name: count, dtype: int64

Missing values in selected features:
search_volume             2468
competition               2468
cpc                       2468
word_count                7699
char_count                7699
content_age_days             0
days_since_last_update       0
content_type                 0
main_intent               2374
competition_level         2610
dtype: int64

Feature vector created successfully.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

The initial feature vector contains seven numeric features and three categorical features.

| Feature | Meaning | Missing-value handling | Available before prediction? |
|---|---|---|---|
| `search_volume` | Estimated search demand | Training-set median | Likely, but source timing needs confirmation |
| `competition` | Keyword competition score | Training-set median | Likely, but source timing needs confirmation |
| `cpc` | Cost per click | Training-set median | Likely, but source timing needs confirmation |
| `word_count` | Number of words in content | Training-set median | Likely, if the content snapshot is available |
| `char_count` | Number of characters in content | Training-set median | Likely, if the content snapshot is available |
| `content_age_days` | Content age in days | Training-set median | Requires a defined prediction date |
| `days_since_last_update` | Days since the last update | Training-set median | Requires a defined prediction date |
| `content_type` | Content category | Most frequent training-set value; one-hot encoding | Likely, if known before prediction |
| `main_intent` | Main search intent | Most frequent training-set value; one-hot encoding | Likely, if known before prediction |
| `competition_level` | Categorical competition level | Most frequent training-set value; one-hot encoding | Likely, but source timing needs confirmation |

These are candidate features. Availability before prediction has not been fully verified because the dataset documentation does not yet establish when every field was recorded.

In [6]:
# ML-05 — Section 2: Handle missing values and categories

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Split before fitting preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# Numeric features: fill missing values with training-set medians
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

# Categorical features: fill missing values and one-hot encode
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

# Learn preprocessing only from training data
X_train_vector = preprocessor.fit_transform(X_train)
X_test_vector = preprocessor.transform(X_test)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("Training vector shape:", X_train_vector.shape)
print("Test vector shape:", X_test_vector.shape)

print(
    "Training values all finite:",
    bool(np.isfinite(X_train_vector).all())
)
print(
    "Test values all finite:",
    bool(np.isfinite(X_test_vector).all())
)

Training rows: 24000
Test rows: 6000
Training vector shape: (24000, 17)
Test vector shape: (6000, 17)
Training values all finite: True
Test values all finite: True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-05 — Step 3: Leakage audit

excluded_fields = {
    "content_id": "Identifier; not used as a predictive feature.",
    "client_id": "Client identifier; excluded for privacy and generalization.",
    "trend_direction": "Target label; must not be an input.",
    "trend_pct": "Directly related to the target trend; excluded.",
    "impressions_90d": "Performance metric; timing and target overlap need verification.",
    "clicks_90d": "Performance metric; timing and target overlap need verification.",
    "pageviews_90d": "Performance metric; timing and target overlap need verification.",
    "sessions_90d": "Performance metric; timing and target overlap need verification.",
    "users_90d": "Performance metric; timing and target overlap need verification.",
    "engaged_sessions_90d": "Performance metric; timing and target overlap need verification.",
    "ai_sessions_90d": "Performance metric; timing and target overlap need verification.",
    "scroll_events_90d": "Performance metric; timing and target overlap need verification.",
    "impressions_last_30d": "Recent-period metric; may be used to calculate the target.",
    "clicks_last_30d": "Recent-period metric; may overlap with the target period.",
    "sessions_last_30d": "Recent-period metric; may overlap with the target period.",
    "impressions_prev_30d": "Comparison-period metric; may be used to calculate the target.",
    "clicks_prev_30d": "Comparison-period metric; may overlap with target calculation.",
    "sessions_prev_30d": "Comparison-period metric; may overlap with target calculation.",
    "ctr": "Derived performance metric; timing and target overlap need verification.",
    "avg_position": "Search performance metric; timing needs verification.",
    "engagement_rate": "Derived performance metric; timing needs verification.",
    "scroll_rate": "Derived performance metric; timing needs verification.",
    "ai_traffic_pct": "Derived performance metric; timing needs verification.",
    "age_tier": "Redundant category derived from content age.",
    "age_tier_order": "Redundant numeric encoding of content age tier.",
    "freshness_tier": "Redundant category derived from update recency.",
    "word_count_tier": "Redundant category derived from word count.",
    "char_count_tier": "Redundant category derived from character count.",
    "impression_tier": "Derived from performance metrics.",
    "position_tier": "Derived from search position metrics.",
    "provider_used": "Excluded from this baseline due to high missingness and possible workflow-specific signal.",
    "model_used": "Excluded from this baseline due to high missingness and possible workflow-specific signal.",
}

print("Excluded fields and reasons:")
for field, reason in excluded_fields.items():
    if field in df.columns:
        print(f"- {field}: {reason}")

# Automated checks on the selected feature table
forbidden_features = {
    "content_id", "client_id", "trend_direction", "trend_pct"
}

assert not (forbidden_features & set(X.columns)), (
    "Leakage/privacy check failed: forbidden fields found in X."
)

assert set(numeric_features + categorical_features) == set(X.columns)

print("\nLeakage exclusion checks passed.")
print("Note: these checks confirm the excluded columns are absent;")
print("they do not prove that every remaining feature is leakage-free.")



Excluded fields and reasons:
- content_id: Identifier; not used as a predictive feature.
- client_id: Client identifier; excluded for privacy and generalization.
- trend_direction: Target label; must not be an input.
- trend_pct: Directly related to the target trend; excluded.
- impressions_90d: Performance metric; timing and target overlap need verification.
- clicks_90d: Performance metric; timing and target overlap need verification.
- pageviews_90d: Performance metric; timing and target overlap need verification.
- sessions_90d: Performance metric; timing and target overlap need verification.
- users_90d: Performance metric; timing and target overlap need verification.
- engaged_sessions_90d: Performance metric; timing and target overlap need verification.
- ai_sessions_90d: Performance metric; timing and target overlap need verification.
- scroll_events_90d: Performance metric; timing and target overlap need verification.
- impressions_last_30d: Recent-period metric; may be used t

In [8]:
# ML-05 — Explicit leakage and privacy tests

# The target and identifiers must not appear in X
assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns
assert "content_id" not in X.columns
assert "client_id" not in X.columns

# Check the excluded trend-related field
assert "trend_pct" not in feature_columns

# Check that the current baseline does not use traffic metrics
traffic_fields = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

traffic_features_in_X = set(traffic_fields).intersection(X.columns)
assert not traffic_features_in_X

# Check that preprocessing preserved the train/test row counts
assert X_train_vector.shape[0] == len(y_train)
assert X_test_vector.shape[0] == len(y_test)

# Check that preprocessing did not leave missing or infinite values
assert np.isfinite(X_train_vector).all()
assert np.isfinite(X_test_vector).all()

print("All explicit leakage and preprocessing checks passed.")
print("Temporal leakage still requires verification of source definitions.")

All explicit leakage and preprocessing checks passed.
Temporal leakage still requires verification of source definitions.


### Leakage checks

- `trend_direction` is the target and must not appear in the feature vector.
- `trend_pct` is excluded because it directly describes the trend outcome.
- Traffic metrics covering recent and previous periods are excluded because they may overlap with the target calculation.
- Tier fields are excluded where they duplicate or summarize other features.
- Client and content identifiers are excluded to reduce privacy exposure and the risk of memorizing identifiers.

The automated checks below verify that the target and identifiers are absent from the selected features. They do not prove that all remaining features are free of temporal leakage. The target-generation logic and feature timestamps must be checked to establish that definitively.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields

- `content_id`: content identifier; excluded from model inputs.
- `client_id`: client identifier; excluded for privacy and generalization.
- `trend_direction`: used as the proxy target, not as an input feature.
- `trend_pct`: directly related to the target trend.
- `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`: performance metrics excluded until their timing relative to the target is verified.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`: recent and comparison-period metrics that may overlap with target construction.
- `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`: derived performance metrics excluded pending timing and target-overlap checks.
- `age_tier`, `age_tier_order`: redundant with content age.
- `freshness_tier`: redundant with update recency.
- `word_count_tier`, `char_count_tier`: redundant with word and character counts.
- `impression_tier`, `position_tier`: derived from performance metrics.
- `provider_used`, `model_used`: excluded from this baseline because of high missingness and possible workflow-specific signals.

This is a conservative baseline for decision-support. Exclusion does not mean these fields can never be useful; they could be reconsidered after their definitions, availability, and leakage risks are verified.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-05 — Section 4: Final feature inventory

print("Selected feature count:", len(X.columns))
print("Selected features:", X.columns.tolist())

print("\nExcluded fields present in dataset:")
print(sorted(set(excluded_fields).intersection(df.columns)))

print("\nIdentifier and target fields excluded from X:")
print(
    sorted(
        {"content_id", "client_id", "trend_direction", "trend_pct"}
        .intersection(df.columns)
    )
)


Selected feature count: 10
Selected features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'content_type', 'main_intent', 'competition_level']

Excluded fields present in dataset:
['age_tier', 'age_tier_order', 'ai_sessions_90d', 'ai_traffic_pct', 'avg_position', 'char_count_tier', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d', 'client_id', 'content_id', 'ctr', 'engaged_sessions_90d', 'engagement_rate', 'freshness_tier', 'impression_tier', 'impressions_90d', 'impressions_last_30d', 'impressions_prev_30d', 'model_used', 'pageviews_90d', 'position_tier', 'provider_used', 'scroll_events_90d', 'scroll_rate', 'sessions_90d', 'sessions_last_30d', 'sessions_prev_30d', 'trend_direction', 'trend_pct', 'users_90d', 'word_count_tier']

Identifier and target fields excluded from X:
['client_id', 'content_id', 'trend_direction', 'trend_pct']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.